# Module 6: AI Agents
## Topics Covered
- AI Agent Fundamentals
- Function Calling & Tool Integration
- Chaining & Tool Orchestration
- Manual Tool Call Parsing
- LangChain Agents: DataFrame Agents, SQL Agents
- Natural Language Database Querying
- AI-Based Data Visualization

---
**Real-world scenario**: Building a data analyst agent that can query databases, analyze CSVs, create visualizations, and fetch external data — all from natural language instructions.

In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-experimental
%pip install -q pandas sqlalchemy matplotlib seaborn tabulate

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-api-key-here')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print('Ready:', OPENAI_API_KEY != 'your-api-key-here')

---
## 1. AI Agent Fundamentals

An **AI Agent** is an LLM that can:
1. **Plan** — decide what actions to take
2. **Use tools** — execute functions to interact with external systems
3. **Observe** — process tool outputs
4. **Iterate** — repeat until task is complete

```
Agent Loop:
User Query → LLM thinks → Calls tool → Gets result → LLM thinks → ... → Final Answer

Example:
"What's the revenue trend for Q4?" →
  1. get_sales_data(period='Q4') →
  2. calculate_trend(data) →
  3. create_chart(trend) →
  Answer: 'Revenue grew 23% in Q4...'
```

---
## 2. Function Calling (OpenAI native)

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

# Define tools as JSON schemas
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_sales_data',
            'description': 'Get sales data for a specific time period and region',
            'parameters': {
                'type': 'object',
                'properties': {
                    'period': {'type': 'string', 'description': 'Time period: Q1, Q2, Q3, Q4, or year'},
                    'region': {'type': 'string', 'enum': ['north', 'south', 'east', 'west', 'all']}
                },
                'required': ['period']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': 'Get current weather for a city',
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {'type': 'string', 'description': 'City name'},
                    'units': {'type': 'string', 'enum': ['celsius', 'fahrenheit'], 'default': 'fahrenheit'}
                },
                'required': ['city']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'calculate_metrics',
            'description': 'Calculate business metrics from sales data',
            'parameters': {
                'type': 'object',
                'properties': {
                    'data': {'type': 'array', 'items': {'type': 'number'}},
                    'metric': {'type': 'string', 'enum': ['mean', 'growth', 'total', 'trend']}
                },
                'required': ['data', 'metric']
            }
        }
    }
]

# Mock function implementations
def get_sales_data(period: str, region: str = 'all') -> dict:
    """Simulate fetching sales from database"""
    np.random.seed(42)
    base = {'Q1': 450000, 'Q2': 520000, 'Q3': 480000, 'Q4': 680000}.get(period, 500000)
    multiplier = {'north': 0.3, 'south': 0.25, 'east': 0.2, 'west': 0.25, 'all': 1.0}.get(region, 1.0)
    values = [base * multiplier * (1 + 0.05 * i + np.random.normal(0, 0.02)) for i in range(3)]
    return {
        'period': period, 'region': region,
        'monthly_revenue': [round(v, 2) for v in values],
        'total': round(sum(values), 2)
    }

def get_weather(city: str, units: str = 'fahrenheit') -> dict:
    """Simulate weather API"""
    import random
    temps = {'New York': (68, 55), 'Los Angeles': (82, 65), 'Chicago': (62, 48), 'Austin': (85, 70)}
    high, low = temps.get(city, (72, 58))
    return {'city': city, 'temp_high': high, 'temp_low': low, 'units': units,
            'condition': random.choice(['Sunny', 'Partly Cloudy', 'Clear'])}

def calculate_metrics(data: list, metric: str) -> dict:
    """Calculate business metrics"""
    if metric == 'mean': return {'metric': 'mean', 'value': round(np.mean(data), 2)}
    if metric == 'total': return {'metric': 'total', 'value': round(sum(data), 2)}
    if metric == 'growth':
        growth = ((data[-1] - data[0]) / data[0]) * 100
        return {'metric': 'growth_pct', 'value': round(growth, 2)}
    if metric == 'trend':
        direction = 'up' if data[-1] > data[0] else 'down'
        return {'metric': 'trend', 'direction': direction, 'delta': round(data[-1] - data[0], 2)}

# Map function names to implementations
AVAILABLE_FUNCTIONS = {
    'get_sales_data': get_sales_data,
    'get_weather': get_weather,
    'calculate_metrics': calculate_metrics
}

print('Tools defined. Ready for function calling.')

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """
    Run the agent loop:
    1. LLM decides which tool to call
    2. Execute the tool
    3. Feed result back to LLM
    4. Repeat until done
    """
    messages = [
        {'role': 'system', 'content': 'You are a data analyst assistant. Use the available tools to answer questions accurately. Always show calculations.'},
        {'role': 'user', 'content': user_query}
    ]
    
    print(f'User: {user_query}')
    print('='*60)
    
    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            tools=tools,
            tool_choice='auto'
        )
        
        message = response.choices[0].message
        messages.append(message.model_dump())
        
        # Check if done
        if response.choices[0].finish_reason == 'stop':
            print(f'Agent Answer: {message.content}')
            return message.content
        
        # Execute tool calls
        if response.choices[0].finish_reason == 'tool_calls':
            for tool_call in message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)
                
                print(f'[Iter {iteration+1}] Calling: {func_name}({func_args})')
                
                # Execute the function
                if func_name in AVAILABLE_FUNCTIONS:
                    result = AVAILABLE_FUNCTIONS[func_name](**func_args)
                    print(f'  Result: {result}')
                else:
                    result = {'error': f'Unknown function: {func_name}'}
                
                # Add tool result to messages
                messages.append({
                    'role': 'tool',
                    'tool_call_id': tool_call.id,
                    'content': json.dumps(result)
                })
    
    return 'Max iterations reached'

# Test agent with multi-step queries
print('\n' + '='*60)
run_agent('What were our Q4 sales and how does that compare to Q1?')

In [ ]:
# Multi-tool query
print('\n' + '='*60)
run_agent('Calculate the growth rate for Q4 sales in the north region')

---
## 3. LangChain Agents

LangChain wraps the agent loop in a cleaner abstraction using `@tool` decorators.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Define tools with @tool decorator
@tool
def fetch_stock_price(ticker: str) -> str:
    """Fetch the current stock price for a given ticker symbol."""
    # Simulated prices
    prices = {'AAPL': 182.50, 'GOOGL': 175.20, 'MSFT': 415.30, 'AMZN': 188.40, 'NVDA': 875.60}
    price = prices.get(ticker.upper(), 'N/A')
    return f'{ticker.upper()}: ${price}'

@tool
def get_company_info(company_name: str) -> str:
    """Get basic information about a company including sector and market cap."""
    companies = {
        'AAPL': 'Apple Inc. | Sector: Technology | Market Cap: $2.8T | PE: 28.5',
        'GOOGL': 'Alphabet Inc. | Sector: Technology | Market Cap: $2.2T | PE: 25.1',
        'MSFT': 'Microsoft Corp | Sector: Technology | Market Cap: $3.1T | PE: 35.2',
        'NVDA': 'NVIDIA Corp | Sector: Semiconductors | Market Cap: $2.1T | PE: 65.4',
    }
    return companies.get(company_name.upper(), f'Company info not found for {company_name}')

@tool
def calculate_portfolio_value(holdings: str) -> str:
    """
    Calculate total portfolio value from holdings string.
    Input format: 'AAPL:10,MSFT:5,NVDA:3' (ticker:shares)
    """
    prices = {'AAPL': 182.50, 'GOOGL': 175.20, 'MSFT': 415.30, 'AMZN': 188.40, 'NVDA': 875.60}
    total = 0
    breakdown = []
    for item in holdings.split(','):
        ticker, shares = item.strip().split(':')
        price = prices.get(ticker.upper(), 0)
        value = price * int(shares)
        total += value
        breakdown.append(f'  {ticker}: {shares} shares × ${price} = ${value:,.2f}')
    return f'Portfolio breakdown:\n' + '\n'.join(breakdown) + f'\n  TOTAL: ${total:,.2f}'

@tool
def search_news(topic: str) -> str:
    """Search for recent news about a topic (simulated)."""
    news = {
        'NVDA': 'NVIDIA announces next-gen Blackwell GPU architecture with 2x performance gains',
        'AAPL': 'Apple Vision Pro sales exceed expectations, new AI features in iOS 18 preview',
        'AI': 'AI investments reach record $50B in Q1 2024, enterprise adoption accelerating',
    }
    return news.get(topic.upper(), f'Recent news: {topic} showing strong market performance and continued growth.')

# Create LangChain agent
agent_tools = [fetch_stock_price, get_company_info, calculate_portfolio_value, search_news]

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a financial analyst assistant. Use tools to research stocks and provide actionable insights. Always cite your data sources.'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}')
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=5)

# Run queries
result = executor.invoke({'input': 'I own 10 AAPL, 5 MSFT, and 3 NVDA shares. What is my portfolio worth? Also get latest NVDA news.'})
print('\nFinal Answer:', result['output'])

---
## 4. DataFrame Agent — Natural Language Data Analysis

In [ ]:
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

# Create sample sales dataset
np.random.seed(42)
n = 200
sales_df = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=n, freq='D'),
    'product': np.random.choice(['Laptop', 'Phone', 'Tablet', 'Headphones', 'Camera'], n),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'salesperson': np.random.choice(['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'], n),
    'units_sold': np.random.randint(1, 20, n),
    'unit_price': np.random.choice([299, 599, 799, 1099, 1499, 1999], n),
    'customer_rating': np.round(np.random.uniform(3.0, 5.0, n), 1)
})
sales_df['revenue'] = sales_df['units_sold'] * sales_df['unit_price']
sales_df['month'] = sales_df['date'].dt.month_name()

print(f'Dataset: {len(sales_df)} rows, {len(sales_df.columns)} columns')
print(sales_df.head())

In [ ]:
# Create DataFrame agent
df_agent = create_pandas_dataframe_agent(
    ChatOpenAI(model='gpt-4o-mini', temperature=0),
    sales_df,
    verbose=True,
    agent_type='tool-calling',  # Use 'openai-tools' for newer versions
    allow_dangerous_code=True   # Required flag for code execution
)

# Natural language data queries
queries = [
    'Which product has the highest total revenue?',
    'Who is the top performing salesperson by units sold?',
    'What is the average customer rating by region?',
]

for query in queries:
    print(f'\nQuery: {query}')
    result = df_agent.invoke({'input': query})
    print(f'Answer: {result["output"]}')
    print('-'*60)

---
## 5. SQL Agent — Natural Language Database Querying

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from sqlalchemy import create_engine, text
import sqlite3

# Create SQLite database from our sales data
engine = create_engine('sqlite:///./sales.db')
sales_df.to_sql('sales', engine, if_exists='replace', index=False)

# Add a customers table
customers_df = pd.DataFrame({
    'customer_id': range(1, 51),
    'name': [f'Customer_{i}' for i in range(1, 51)],
    'region': np.random.choice(['North', 'South', 'East', 'West'], 50),
    'tier': np.random.choice(['Gold', 'Silver', 'Bronze'], 50),
    'lifetime_value': np.random.randint(500, 50000, 50)
})
customers_df.to_sql('customers', engine, if_exists='replace', index=False)

# Connect LangChain SQL interface
db = SQLDatabase(engine)
print('Tables:', db.get_usable_table_names())
print('\nSales schema:')
print(db.get_table_info(['sales']))

In [ ]:
# Create SQL agent
sql_agent = create_sql_agent(
    llm=ChatOpenAI(model='gpt-4o-mini', temperature=0),
    db=db,
    verbose=True,
    agent_type='tool-calling'
)

# Natural language SQL queries
nl_queries = [
    'What are the top 3 products by total revenue?',
    'Which region has the most Gold tier customers?',
    'What is the average revenue per sale for each product category?'
]

for q in nl_queries:
    print(f'\nNL Query: {q}')
    result = sql_agent.invoke({'input': q})
    print(f'Answer: {result["output"]}')
    print('-'*60)

---
## 6. AI-Based Data Visualization Agent

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from langchain.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor

@tool
def create_bar_chart(title: str, x_label: str, y_label: str, data_query: str) -> str:
    """
    Create a bar chart from sales data.
    data_query options: 'revenue_by_product', 'revenue_by_region', 'units_by_salesperson'
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    if data_query == 'revenue_by_product':
        data = sales_df.groupby('product')['revenue'].sum().sort_values(ascending=False)
    elif data_query == 'revenue_by_region':
        data = sales_df.groupby('region')['revenue'].sum().sort_values(ascending=False)
    elif data_query == 'units_by_salesperson':
        data = sales_df.groupby('salesperson')['units_sold'].sum().sort_values(ascending=False)
    else:
        return f'Unknown data query: {data_query}'
    
    colors = sns.color_palette('husl', len(data))
    bars = ax.bar(data.index, data.values, color=colors)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    
    # Add value labels on bars
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'${val:,.0f}' if 'revenue' in data_query else f'{val:,}',
                ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    filename = f'chart_{data_query}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    return f'Chart saved as {filename}. Top result: {data.index[0]} with {data.values[0]:,.0f}'

@tool
def create_trend_chart(metric: str, groupby_period: str) -> str:
    """
    Create a line chart showing trends over time.
    metric: 'revenue' or 'units_sold'
    groupby_period: 'month' or 'week'
    """
    fig, ax = plt.subplots(figsize=(12, 5))
    
    if groupby_period == 'month':
        trend_data = sales_df.groupby('month')[metric].sum()
        month_order = ['January', 'February', 'March', 'April', 'May', 'June',
                       'July', 'August', 'September', 'October', 'November', 'December']
        trend_data = trend_data.reindex([m for m in month_order if m in trend_data.index])
    else:
        sales_df['week'] = sales_df['date'].dt.isocalendar().week
        trend_data = sales_df.groupby('week')[metric].sum()
    
    ax.plot(trend_data.index, trend_data.values, marker='o', linewidth=2, markersize=5)
    ax.fill_between(range(len(trend_data)), trend_data.values, alpha=0.2)
    ax.set_xticks(range(len(trend_data)))
    ax.set_xticklabels(trend_data.index, rotation=45)
    ax.set_title(f'{metric.title()} Trend by {groupby_period.title()}', fontsize=14)
    ax.set_ylabel(metric.title())
    plt.tight_layout()
    plt.savefig(f'trend_{metric}.png', dpi=150)
    plt.show()
    return f'Trend chart created. Min: {trend_data.min():,.0f}, Max: {trend_data.max():,.0f}'

@tool
def get_summary_stats(column: str) -> str:
    """Get summary statistics for a sales data column. Valid columns: revenue, units_sold, customer_rating, unit_price"""
    if column not in sales_df.columns:
        return f'Column {column} not found. Available: {list(sales_df.select_dtypes(include=[np.number]).columns)}'
    stats = sales_df[column].describe()
    return stats.to_string()

# Visualization agent
viz_tools = [create_bar_chart, create_trend_chart, get_summary_stats]

viz_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a data visualization expert. Create insightful charts and provide analysis.'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}')
])

viz_agent = create_tool_calling_agent(llm, viz_tools, viz_prompt)
viz_executor = AgentExecutor(agent=viz_agent, tools=viz_tools, verbose=True)

# Ask agent to create visualizations
result = viz_executor.invoke({'input': 'Create a bar chart showing revenue by product, and also show the monthly revenue trend. What insights can you draw?'})
print('\nInsights:', result['output'])

---
## Summary

| Component | Purpose | API |
|-----------|---------|-----|
| Function Calling | Structured tool invocation | OpenAI `tools` parameter |
| Agent Loop | Plan → Act → Observe → Repeat | `finish_reason='tool_calls'` |
| LangChain @tool | Decorator-based tool definition | `from langchain.tools import tool` |
| DataFrame Agent | Natural language CSV/DF analysis | `create_pandas_dataframe_agent` |
| SQL Agent | Natural language DB queries | `create_sql_agent` |
| Viz Agent | Auto chart generation | Custom tools + AgentExecutor |

### Next: Module 7 — LangGraph for stateful, multi-agent workflows